# Bakalar et al. (2023), dandiset 001362: FRST thresholds 14000 to 26000

Part of the entry `001362-bakalar-2023`; the main notebook, `reproduce_bakalar_2023.ipynb`, has the
paper, the dataset, the data checks and the claims table. The claims file (fig5-dg-fos, reading 1)
asks for every FRST threshold the derivatives hold. The main notebook runs the five that all 22
hemispheres hold (4000 to 12000). Thresholds 14000 to 40000 exist for 18 hemispheres only, the 4
others having been segmented on a finer series (2000 to 12000). Counting all 14 in one notebook
would take over the 60-minute limit, so this notebook runs 14000 to 26000. Thresholds 28000 to
40000 were not run: the run that should have counted them was stopped for time. The counting, the regions and the tests are those of the main notebook (`fos.py`), and so
are the readings: region size (density or raw count), the IQR rule (within each genotype or
pooled), the percentage (ratio of means or of medians), both counting rules (9 to 900 voxels, the
main one, and every component). The recount takes about 35 minutes; no random numbers are drawn.

| Claim | Status here |
|---|---|
| fig5-dg-fos | tested at thresholds 14000 to 26000, 18 hemispheres |
| table4-lenient-regions | tested at thresholds 14000 to 26000, 18 hemispheres |
| table4-region-values | tested at thresholds 14000 to 26000, 18 hemispheres |
| the other 24 claims | blocked: data missing (see the main notebook) |

In [ ]:
import time, warnings
import numpy as np
import pandas as pd
import fos

warnings.filterwarnings("ignore", category=RuntimeWarning)
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 200)
HIGH = fos.HIGH_THRESHOLDS[:7]
print("dandiset", fos.DANDISET_ID, "version", fos.DANDISET_VERSION, "| thresholds run here:", HIGH)

## Data check

In [ ]:
import json
S = fos.samples()
S["holds"] = [all(t in r.seg_thresholds for t in HIGH) for r in S.itertuples()]
print(S[["subject", "hemisphere", "genotype", "atlas_masked", "count_tables", "holds"]].to_string())
H = S[S.holds].reset_index(drop=True)
hemis = list(zip(H.subject, H.hemisphere))
geno = H.genotype
print("\nhemispheres holding every threshold run here:", len(H), "| by genotype:", geno.value_counts().to_dict(),
      "| mice:", H.groupby("genotype").subject.nunique().to_dict())
print("left out (finer series 2000 to 12000):", [f"{r.subject} {r.hemisphere}" for r in S[~S.holds].itertuples()])
spim = json.loads(fos.read_text("SPIM.json"))
VOX_MM3 = np.prod(spim["PixelSize"]) * 1e-9
print("voxel volume (mm^3): %.3e" % VOX_MM3)
R = fos.regions()
inc = R[R.included].reset_index(drop=True)
print("regions included:", len(inc))

In [ ]:
t0 = time.time()
RC = fos.count_all(hemis, HIGH, workers=2)
print("recount took %.1f min" % ((time.time() - t0) / 60))
rows = []
for (s, h), r in RC.items():
    for t in HIGH:
        n_all, n_sized, n_small, n_big = r["n_components"][t]
        rows.append(dict(subject=s, hemisphere=h, threshold=t, components=n_all, within_9_900=n_sized,
                         below_9=n_small, above_900=n_big, too_tall_pieces_dropped=r["dropped"][t]))
comp = pd.DataFrame(rows)
print(comp.groupby("threshold")[["components", "within_9_900", "below_9", "above_900", "too_tall_pieces_dropped"]].sum())
print("pieces of components too tall to measure, as a fraction of all components: %.2e" % (comp.too_tall_pieces_dropped.sum() / comp.components.sum()))

In [ ]:
# Check the recount against the dataset's count tables, for the hemispheres that have them
rows = []
for r in H[H.count_tables].itertuples():
    rc = RC[(r.subject, r.hemisphere)]
    tabs = fos.count_table_paths(r.subject, r.hemisphere)
    for t in HIGH:
        if t not in tabs:
            continue
        tab = fos.read_tsv(tabs[t])
        mine_all = rc["all"][t][tab.label_id.values]
        rows.append(dict(subject=r.subject, hemisphere=r.hemisphere, threshold=t, table_total=int(tab.cell_count.sum()),
                         recount_all=int(mine_all.sum()), recount_9_900=int(rc["sized"][t][tab.label_id.values].sum()),
                         abs_diff_all_pct=100 * np.abs(mine_all - tab.cell_count.values).sum() / tab.cell_count.sum(),
                         volume_equal=bool((rc["volume"][tab.label_id.values] == tab.label_volume_pixels.values).all())))
check = pd.DataFrame(rows)
print("count tables compared:", len(check), "from", check.groupby(["subject", "hemisphere"]).ngroups, "hemispheres")
print("summed absolute per-region difference, all components against the tables: median %.2f%%, max %.2f%%"
      % (check.abs_diff_all_pct.median(), check.abs_diff_all_pct.max()))
print("components of 9 to 900 voxels as a fraction of the table totals: %.2f to %.2f"
      % ((check.recount_9_900 / check.table_total).min(), (check.recount_9_900 / check.table_total).max()))
print("region volumes equal to the tables' label_volume_pixels in every row:", check.volume_equal.all())

In [ ]:
def matrix(t, rule="sized", size="density"):
    rows = []
    for s, h in hemis:
        r = RC[(s, h)]
        c = r[rule][t][inc.label_id.values].astype(float)
        v = r["volume"][inc.label_id.values].astype(float)
        rows.append(np.where(v > 0, c / (v * VOX_MM3) if size == "density" else c, np.nan))
    m = pd.DataFrame(rows, columns=inc.acronym.values)
    return m.loc[:, m.notna().any()]

GRID = [(t, size, iqr) for t in HIGH for size in ("density", "raw") for iqr in ("within", "pooled")]
TESTS = {}
for rule in ("sized", "all"):
    for t, size, iqr in GRID:
        TESTS[(rule, t, size, iqr)] = fos.region_tests(matrix(t, rule, size), geno, iqr)
print("analyses run:", len(TESTS), "| regions tested per analysis:", sorted({len(v) for v in TESTS.values()}))

## fig5-dg-fos

> In whole-brain c-Fos counts after the early dark phase, PACAPko hemispheres have a higher density
> of c-Fos-positive cells than PACAP fl/fl in the dentate gyrus granule cell layer and polymorph
> layer, the only two regions significant after correction for multiple testing; PACAPko densities
> are 224% and 326% of control.

Criterion (claims file): reproduced when DG-sg and DG-po both have PACAPko higher with BKY-adjusted
p < 0.05; partially reproduced when one of the two holds; blocked when the readings of (1) to (4)
give different outcomes. With 10 PACAPko and 8 PACAP fl/fl hemispheres, U can reach 80 (the paper
88, at 11 and 8).

In [ ]:
def dg_outcome(tab):
    ok = [(r in tab.index) and tab.loc[r, "p_adj"] < 0.05 and tab.loc[r, "mean_ko"] > tab.loc[r, "mean_flox"] for r in ("DG-sg", "DG-po")]
    return "reproduced" if all(ok) else ("partial" if any(ok) else "not")

rows = []
for rule in ("sized", "all"):
    for t, size, iqr in GRID:
        tab = TESTS[(rule, t, size, iqr)]
        d = dict(rule=rule, threshold=t, size=size, iqr=iqr)
        for r in ("DG-sg", "DG-po"):
            x = tab.loc[r]
            d.update({f"{r} n": f"{int(x.n_ko)}/{int(x.n_flox)}", f"{r} U": x.U, f"{r} p": x.p, f"{r} p_adj": x.p_adj,
                      f"{r} %mean": x.pct_mean, f"{r} %median": x.pct_median})
        d["n regions p_adj<0.05"] = int((tab.p_adj < 0.05).sum())
        d["outcome"] = dg_outcome(tab)
        rows.append(d)
dg = pd.DataFrame(rows)
print(dg.to_string(float_format=lambda v: "%.4g" % v))
print("\noutcomes by counting rule:"); print(dg.groupby("rule").outcome.value_counts().unstack(fill_value=0).to_string())
print("\nPACAPko higher (ratio of means) in DG-sg in %d of %d analyses, in DG-po in %d"
      % ((dg["DG-sg %mean"] > 100).sum(), len(dg), (dg["DG-po %mean"] > 100).sum()))
print("DG-sg raw p %.2e to %.2e; DG-po raw p %.2e to %.2e" % (dg["DG-sg p"].min(), dg["DG-sg p"].max(), dg["DG-po p"].min(), dg["DG-po p"].max()))

**Result: the higher thresholds do not reproduce the claim.** On the 18 hemispheres that hold
thresholds 14000 to 26000 (8 PACAP fl/fl from 6 mice, 10 PACAPko from 6 mice), PACAPko is higher
in DG-sg and DG-po in all 56 analyses (both counting rules, 7 thresholds, two region sizes, two
outlier rules). In 55 of them neither region has BKY-adjusted p < 0.05; in the other one
(9 to 900 voxels, 22000, raw counts, within-genotype outliers), one of the two does. Raw p is
5.76e-04 to 0.114 for DG-sg and 1.75e-04 to 0.046 for DG-po. Together with the main notebook,
where the readings at 4000 to 12000 also disagree, the readings give different outcomes.

## table4-lenient-regions

> At an uncorrected cutoff of raw p <= 0.01, 13 further regions differ in c-Fos cell density between
> PACAPko and PACAP fl/fl, including substantia nigra reticular part, subthalamic nucleus and
> intergeniculate leaflet.

Criterion: reproduced when SNr, STN and IGL have raw p <= 0.01 with PACAPko higher and the number of
other regions at raw p <= 0.01 is within 13 +/- 1.3; partially reproduced when one or two of the
three meet the cutoff.

In [ ]:
T4 = ["DG-sg", "DG-po", "LGv", "IGL", "SNr", "AUDp2/3", "DTN", "IO", "AUDpo2/3", "LGd", "PST", "LRN", "STN", "PGRNl", "NB"]
def lenient(tab):
    three = {r: bool(r in tab.index and tab.loc[r, "p"] <= 0.01 and tab.loc[r, "mean_ko"] > tab.loc[r, "mean_flox"]) for r in ("SNr", "STN", "IGL")}
    hits = tab[tab.p <= 0.01].index
    n_other = int(len([r for r in hits if r not in ("DG-sg", "DG-po")]))
    k = sum(three.values())
    out = "reproduced" if k == 3 and abs(n_other - 13) <= 1.3 else ("partial" if k > 0 else "not")
    return three, n_other, len(set(hits) & set(T4)), out

rows = []
for (rule, t, size, iqr), tab in TESTS.items():
    three, n, ov, out = lenient(tab)
    rows.append(dict(rule=rule, threshold=t, size=size, iqr=iqr, **{f"{r} p": tab.loc[r, "p"] for r in ("SNr", "STN", "IGL")},
                     **{f"{r} ok": v for r, v in three.items()}, other_regions_p01=n, overlap_with_table4_of_15=ov, outcome=out))
ln = pd.DataFrame(rows)
print(ln.to_string(float_format=lambda v: "%.3g" % v))
print("\noutcomes by counting rule:"); print(ln.groupby("rule").outcome.value_counts().unstack(fill_value=0).to_string())
print("\nmeeting the cutoff with PACAPko higher, of %d analyses: SNr %d, STN %d, IGL %d; other regions at p <= 0.01: %d to %d"
      % (len(ln), ln["SNr ok"].sum(), ln["STN ok"].sum(), ln["IGL ok"].sum(), ln.other_regions_p01.min(), ln.other_regions_p01.max()))

**Result: the higher thresholds give not reproduced in 54 of 56 analyses.** At 14000 to 26000 on the
18 hemispheres, SNr and STN never meet raw p <= 0.01 with PACAPko higher, and IGL meets it in 2 of
56 analyses (every component counted, 14000, density). Those 2 analyses are partial; the other 54
are not reproduced. The number of other regions at raw p <= 0.01 is 4 to 21. The main
notebook's readings at 4000 to 12000 are all partial, so the readings over the thresholds
disagree.

## table4-region-values

> Table 4 gives, for each of the 15 regions with raw p <= 0.01, the Mann-Whitney U and PACAPko density
> as a percentage of PACAP fl/fl; every region but DTN (45%) is higher in PACAPko (167% to 326%).

Criterion: reproduced when, under one reading, the direction matches for all 13 regions and at
least 10 of the 13 have both U and percentage within tolerance; partially reproduced when the
direction matches for all and fewer are within tolerance; not reproduced when any direction
differs.

In [ ]:
EXP = {"LGv": (73, 170.42), "IGL": (72, 258.62), "SNr": (80, 275.91), "AUDp2/3": (73, 261.06), "DTN": (9, 45.21),
       "IO": (76, 297.49), "AUDpo2/3": (76, 198.26), "LGd": (76, 276.46), "PST": (68, 208.73), "LRN": (76, 293.57),
       "STN": (70, 196.47), "PGRNl": (75, 239.16), "NB": (75, 167.26)}
TOL = {"LGv": (7.3, 17.0), "IGL": (7.2, 25.9), "SNr": (8.0, 27.6), "AUDp2/3": (7.3, 26.1), "DTN": (0.9, 4.5),
       "IO": (7.6, 29.7), "AUDpo2/3": (7.6, 19.8), "LGd": (7.6, 27.6), "PST": (6.8, 20.9), "LRN": (7.6, 29.4),
       "STN": (7.0, 19.6), "PGRNl": (7.5, 23.9), "NB": (7.5, 16.7)}
rows = []
for (rule, t, size, iqr), tab in TESTS.items():
    for pct in ("pct_mean", "pct_median"):
        dirs = [(tab.loc[r, pct] > 100) == (EXP[r][1] > 100) for r in EXP]
        within = [abs(tab.loc[r, "U"] - EXP[r][0]) <= TOL[r][0] and abs(tab.loc[r, pct] - EXP[r][1]) <= TOL[r][1] for r in EXP]
        out = ("reproduced" if sum(within) >= 10 else "partial") if all(dirs) else "not"
        rows.append(dict(rule=rule, threshold=t, size=size, iqr=iqr, pct=pct, directions_match=sum(dirs),
                         wrong_direction=" ".join(r for r, d in zip(EXP, dirs) if not d), within_tol=sum(within), outcome=out))
tv = pd.DataFrame(rows)
print(tv.to_string())
print("\noutcomes by counting rule:"); print(tv.groupby("rule").outcome.value_counts().unstack(fill_value=0).to_string())
print("most regions within tolerance in any analysis:", tv.within_tol.max())
tab = TESTS[("sized", HIGH[0], "density", "within")]
print("\nvalues at threshold %d, main rule, density, IQR within genotype:" % HIGH[0])
print(pd.DataFrame({"U": tab.loc[list(EXP), "U"], "U paper": [EXP[r][0] for r in EXP], "% mean": tab.loc[list(EXP), "pct_mean"],
                    "% paper": [EXP[r][1] for r in EXP], "raw p": tab.loc[list(EXP), "p"]}).to_string(float_format=lambda v: "%.4g" % v))

**Result: partial in 110 of 112 analyses, not reproduced in 2.** At 14000 to 26000 on the
18 hemispheres, every analysis but 2 has the direction of all 13 regions as in Table 4. The 2
exceptions (9 to 900 voxels, 26000, raw counts or density, within-genotype outliers, ratio of
medians) have LRN in the wrong direction. At most 3 of the 13 regions have both U and percentage
within tolerance, against the 10 the criterion asks. U can reach 80 here (10 x 8), and DTN has
U = 0 at 14000 against 9 in the paper.

## Summary

| Claim | At 14000 to 26000, 18 hemispheres |
|---|---|
| fig5-dg-fos | not reproduced in 55 of 56 analyses, partial in 1 |
| table4-lenient-regions | not reproduced in 54 of 56, partial in 2 |
| table4-region-values | partial in 110 of 112, not reproduced in 2 |

These results are variants for the threshold reading of the claims file. The outcomes are recorded
in the main notebook and the attempt record. Thresholds 28000 to 40000 were not run: the run that
should have counted them was stopped for time.

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning
and the differences from the paper's method.